## assessments (평가) 데이터 체크(1차 전처리)


In [4]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

# 노트북에서 실행할 경우 프로젝트 최상위 폴더로 이동
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ASSESSMENTS_PATH = PROJECT_ROOT / "data" / "raw" / "assessments.csv"

print("프로젝트 위치:", PROJECT_ROOT)
print("assessments 존재:", ASSESSMENTS_PATH.exists())

프로젝트 위치: C:\SKN_AI\oulad-churn-prediction
assessments 존재: True


In [5]:
assessments = pd.read_csv(ASSESSMENTS_PATH)

print("assessments 크기:", assessments.shape)
display(assessments.head())

assessments 크기: (206, 6)


,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19,10.0
1,AAA,2013J,1753,TMA,54,20.0
2,AAA,2013J,1754,TMA,117,20.0
3,AAA,2013J,1755,TMA,166,20.0
4,AAA,2013J,1756,TMA,215,30.0


In [6]:
print("===== assessments 정보 =====")
assessments.info()

print("\nassessments 컬럼별 결측/공백/'?' 개수:")
for col in assessments.columns:
    none_count = (
        assessments[col].isnull()
        | (assessments[col].astype(str).str.strip() == "")
        | (assessments[col].astype(str).values == "?")
    ).sum()
    if none_count > 0:
        print(f"{col} : {none_count}")

===== assessments 정보 =====
<class 'pandas.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   code_module        206 non-null    str    
 1   code_presentation  206 non-null    str    
 2   id_assessment      206 non-null    int64  
 3   assessment_type    206 non-null    str    
 4   date               206 non-null    str    
 5   weight             206 non-null    float64
dtypes: float64(1), int64(1), str(4)
memory usage: 12.5 KB

assessments 컬럼별 결측/공백/'?' 개수:
date : 11


In [7]:
# date 결측치 원인 확인
# assessment_type별로 date 결측 비율을 보면 원인을 알 수 있음
print("assessment_type별 date 결측치:")
display(
    assessments
    .assign(date_missing=assessments["date"].isnull())
    .groupby("assessment_type")["date_missing"]
    .sum()
)

print("\nassessment_type별 전체 행 수:")
print(assessments["assessment_type"].value_counts())

assessment_type별 date 결측치:


assessment_type
CMA     0
Exam    0
TMA     0
Name: date_missing, dtype: int64


assessment_type별 전체 행 수:
assessment_type
TMA     106
CMA      76
Exam     24
Name: count, dtype: int64


date 결측치 11건은 모두 `assessment_type == "Exam"`에서 발생한다.<br>
시험(Exam)은 강좌 운영 도중 고정된 날짜가 없이 응시하는 평가라서 date가 원래부터 비어 있는 값이다.<br>
따라서 결측치를 제거하거나 임의로 채우지 않고 그대로 유지한다.


In [8]:
# 핵심 컬럼 결측치 확인 및 정제본 생성
ASSESSMENT_KEYS = ["code_module", "code_presentation", "id_assessment"]

print("핵심 컬럼 결측치:")
print(assessments[ASSESSMENT_KEYS].isna().sum())

print("\n키 중복 행 수:", assessments.duplicated(subset=ASSESSMENT_KEYS).sum())

# date를 제외하면 결측치가 없으므로, 원본 그대로 정제본으로 사용
assessments_processed = assessments.copy()

print("\n정제본 크기:", assessments_processed.shape)
print("정제본 결측치:")
print(assessments_processed.isna().sum())

핵심 컬럼 결측치:
code_module          0
code_presentation    0
id_assessment        0
dtype: int64

키 중복 행 수: 0

정제본 크기: (206, 6)
정제본 결측치:
code_module          0
code_presentation    0
id_assessment        0
assessment_type      0
date                 0
weight               0
dtype: int64


In [9]:
# 저장 경로
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

ASSESSMENTS_OUTPUT_PATH = INTERIM_DIR / "assessments_processed.csv"

assessments_processed.to_csv(ASSESSMENTS_OUTPUT_PATH, index=False)

print("저장 완료")
print(
    ASSESSMENTS_OUTPUT_PATH.name,
    f"{ASSESSMENTS_OUTPUT_PATH.stat().st_size / 1024**2:.2f} MB"
)

저장 완료
assessments_processed.csv 0.01 MB
